# 01 — Generación de datos sintéticos (Bronze)

Este notebook implementa la generación descrita en `doc.md`, secciones 3.2–3.5.
Crea cinco CSV sin limpiar para simular las fuentes del pipeline. La base válida contiene
1,000 clientes, 28 productos, 10 restaurantes y 20,000 compras en tres meses.

**Ejecución:** ejecutar todas las celdas en orden. Solo se usa la biblioteca estándar.
La semilla fija permite reproducir los archivos. Las celdas finales introducen errores
controlados para el notebook 02; los datos limpios se validan antes de ese paso.
Los nombres, correos, restaurantes, precios y comportamientos son sintéticos.


In [1]:
import calendar
import csv
import hashlib
import random
from collections import Counter, defaultdict
from datetime import date, datetime, time, timedelta
from decimal import Decimal
from pathlib import Path

# Funciona al iniciar Jupyter desde la raíz del proyecto o desde notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "doc.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Ejecute este notebook desde la carpeta que contiene doc.md o notebooks/.")
BRONZE = PROJECT_ROOT / "data" / "bronze"
SEED = 22434
YEAR = 2025
POINTS_PER_QUETZAL = 10
ANALYSIS_END = date(YEAR, 2, 28)
PERIOD_START = date(YEAR, 1, 1)
PERIOD_END = date(YEAR, 3, 31)
CATEGORIES = (
    "hamburguesas", "pollo", "desayunos", "acompañamientos",
    "bebidas", "postres", "ensaladas",
)
PROFILES = ("Leales", "Madrugadores", "En riesgo", "Ocasionales")
SCHEMAS = {
    "clientes": ["id", "nombre", "correo", "fecha_registro"],
    "productos": ["id", "nombre", "categoria", "precio_normal", "precio_en_puntos"],
    "restaurantes": ["id", "nombre", "zona"],
    "compras": ["id", "cliente_id", "restaurante_id", "fecha_hora", "monto_total", "forma_pago"],
    "lineas_compra": ["id", "compra_id", "producto_id", "cantidad", "precio_unitario", "subtotal", "pagado_con_puntos"],
}


def money(cents):
    return f"{Decimal(cents) / 100:.2f}"


def cents(value):
    amount = Decimal(str(value)) * 100
    assert amount == amount.to_integral_value(), "Monto con más de dos decimales"
    return int(amount)


def base_points(amount_cents):
    return (amount_cents * POINTS_PER_QUETZAL + 99) // 100


assert base_points(1235) == 124
assert base_points(5000) == 500
print(f"Proyecto: {PROJECT_ROOT}")
print(f"Período: {PERIOD_START} a {PERIOD_END}; semilla: {SEED}")


Proyecto: /home/nelson/Documents/Uvg/MLOPS/ML-Engineering/Casos-Estudio/Caso-Estudio-2
Período: 2025-01-01 a 2025-03-31; semilla: 22434


## 1. Clientes, catálogo y restaurantes

Los cuatro perfiles se distribuyen entre los clientes al azar. Las etiquetas ocultas
sirven para construir y revisar el conjunto sintético, pero no forman parte de los CSV.
Cada categoría contiene cuatro productos; cuatro productos de categorías distintas
tienen un peso de selección reducido. Todos los clientes se registran antes del período.


In [2]:
# Reiniciar esta celda reinicia toda la generación determinista.
rng = random.Random(SEED)
customer_ids = list(range(1, 1001))
rng.shuffle(customer_ids)
hidden_profiles = {
    customer_id: PROFILES[index // 250]
    for index, customer_id in enumerate(customer_ids)
}
clients = [
    {
        "id": customer_id,
        "nombre": f"Cliente Sintético {customer_id:04d}",
        "correo": f"cliente{customer_id:04d}@example.com",
        "fecha_registro": (PERIOD_START - timedelta(days=rng.randint(1, 365))).isoformat(),
    }
    for customer_id in sorted(customer_ids)
]
# Precios en centavos; se incluyen fracciones para ejercitar el redondeo.
menu = {
    "hamburguesas": [("Big Mac", 3595), ("Cuarto de libra", 3890), ("Hamburguesa con queso", 1800), ("Hamburguesa especial", 3250)],
    "pollo": [("McPollo", 3200), ("McNuggets 6", 2850), ("Pollo crujiente", 3500), ("Sándwich de pollo especial", 3695)],
    "desayunos": [("McMuffin huevo", 2450), ("McMuffin salchicha", 2695), ("Hotcakes", 2200), ("Desayuno especial", 3100)],
    "acompañamientos": [("Papas pequeñas", 1235), ("Papas medianas", 1650), ("Papas grandes", 1995), ("Papas especiales", 2250)],
    "bebidas": [("Café", 1095), ("Gaseosa", 1200), ("Jugo de naranja", 1550), ("Agua", 1000)],
    "postres": [("McFlurry Oreo", 2495), ("Sundae chocolate", 1800), ("Cono", 800), ("Pie de manzana", 1500)],
    "ensaladas": [("Ensalada de pollo", 3500), ("Ensalada verde", 2600), ("Ensalada mixta", 2950), ("Ensalada especial", 3295)],
}
products = []
product_cents = {}
for category, items in menu.items():
    for name, price in items:
        product_id = len(products) + 1
        products.append({
            "id": product_id, "nombre": name, "categoria": category,
            "precio_normal": money(price), "precio_en_puntos": (price * 60 + 99) // 100,
        })
        product_cents[product_id] = price
laggard_ids = {4, 8, 24, 28}
product_by_id = {product["id"]: product for product in products}
products_by_category = {
    category: [product["id"] for product in products if product["categoria"] == category]
    for category in CATEGORIES
}
restaurants = [
    {"id": index, "nombre": f"Restaurante Sintético {index:02d}", "zona": zone}
    for index, zone in enumerate(
        ["Zona 1", "Zona 4", "Zona 7", "Zona 9", "Zona 10", "Zona 11", "Zona 12", "Zona 15", "Mixco", "Villa Nueva"], 1
    )
]
print(f"Clientes: {len(clients)}; productos: {len(products)}; restaurantes: {len(restaurants)}")
print("Perfiles ocultos:", dict(Counter(hidden_profiles.values())))


Clientes: 1000; productos: 28; restaurantes: 10
Perfiles ocultos: {'Leales': 250, 'Madrugadores': 250, 'En riesgo': 250, 'Ocasionales': 250}


## 2. Calendario de compras y preferencias

Se distribuyen 20,000 compras con variación individual: Leales 9,000;
Madrugadores 6,500; En riesgo 3,500; Ocasionales 1,000.
Los clientes En riesgo concentran sus compras en enero y reducen su actividad
en febrero, cuando solo compran durante la primera semana.
Los Ocasionales realizan una o dos compras por mes. Los Madrugadores compran
principalmente entre las 06:00 y las 11:00. Cada perfil tiene preferencias de categoría.

El calendario se ordena cronológicamente antes de simular saldos y canjes.


In [3]:
monthly_targets = {
    "Leales": [3000, 3000, 3000],
    "Madrugadores": [2166, 2167, 2167],
    "En riesgo": [3000, 250, 250],
    "Ocasionales": [333, 333, 334],
}
# Pesos en el orden definido en CATEGORIES.
category_weights = {
    "Leales": [30, 25, 5, 12, 8, 17, 3],
    "Madrugadores": [3, 2, 51, 3, 35, 3, 3],
    "En riesgo": [20, 15, 8, 8, 8, 8, 33],
    "Ocasionales": [8, 5, 5, 30, 35, 15, 2],
}
# Franjas: mañana [06,11), almuerzo [11,15), tarde [15,18), noche [18,23).
time_windows = [(6, 11), (11, 15), (15, 18), (18, 23)]
hour_weights = {
    "Leales": [25, 25, 25, 25],
    "Madrugadores": [97, 1, 1, 1],
    "En riesgo": [10, 40, 15, 35],
    "Ocasionales": [10, 20, 40, 30],
}
calendar_events = []
for profile in PROFILES:
    members = sorted(customer_id for customer_id in customer_ids if hidden_profiles[customer_id] == profile)
    activity_weights = [rng.uniform(0.4, 1.6) for _ in members]
    for month, target in enumerate(monthly_targets[profile], 1):
        if profile == "Ocasionales":
            visits = members + rng.sample(members, target - len(members))
        else:
            minimum = 4 if profile == "Leales" else 3 if profile == "Madrugadores" or month == 1 else 0
            visits = members * minimum
            visits += rng.choices(members, weights=activity_weights, k=target - len(visits))
        for customer_id in visits:
            max_day = 7 if profile == "En riesgo" and month == 2 else calendar.monthrange(YEAR, month)[1]
            day = rng.randint(1, max_day)
            start_hour, end_hour = rng.choices(time_windows, weights=hour_weights[profile])[0]
            seconds = rng.randrange((end_hour - start_hour) * 3600)
            timestamp = datetime(YEAR, month, day, start_hour) + timedelta(seconds=seconds)
            calendar_events.append((timestamp, customer_id, rng.randint(1, len(restaurants))))
calendar_events.sort()
assert len(calendar_events) == 20_000
print("Calendario listo:", len(calendar_events), "compras")


Calendario listo: 20000 compras


## 3. Líneas de compra y canjes

Cada compra tiene de una a cuatro líneas con productos distintos. Los Leales tienen
más líneas y unidades para simular tickets mayores. Los productos rezagados tienen
peso `0.025`, frente a `1` para sus pares disponibles de la misma categoría.

Se usa dinero en centavos enteros para evitar errores de punto flotante. Los canjes
se permiten solo con el saldo disponible antes de la compra; no generan acumulación.
Se descuentan primero los canjes y se acreditan los puntos monetarios al terminar la
compra. Estos saldos son internos al generador; `movimientos_puntos` corresponde al notebook 05.


In [4]:
purchases = []
lines = []
balances = defaultdict(int)
for purchase_id, (timestamp, customer_id, restaurant_id) in enumerate(calendar_events, 1):
    profile = hidden_profiles[customer_id]
    count_weights = [5, 20, 40, 35] if profile == "Leales" else [35, 40, 20, 5] if profile == "Madrugadores" else [65, 25, 8, 2] if profile == "Ocasionales" else [20, 40, 30, 10]
    number_of_lines = rng.choices([1, 2, 3, 4], weights=count_weights)[0]
    selected = set()
    purchase_total = 0
    points_earned = 0
    for _ in range(number_of_lines):
        category = rng.choices(CATEGORIES, weights=category_weights[profile])[0]
        candidates = [product_id for product_id in products_by_category[category] if product_id not in selected]
        product_id = rng.choices(candidates, weights=[0.025 if pid in laggard_ids else 1 for pid in candidates])[0]
        selected.add(product_id)
        quantity = rng.choices([1, 2, 3], weights=[65, 30, 5])[0] if profile == "Leales" else 1
        redemption_cost = product_by_id[product_id]["precio_en_puntos"] * quantity
        redeemed = balances[customer_id] >= redemption_cost and rng.random() < 0.06
        unit_price = 0 if redeemed else product_cents[product_id]
        subtotal = unit_price * quantity
        if redeemed:
            balances[customer_id] -= redemption_cost
        else:
            points_earned += base_points(subtotal)
        purchase_total += subtotal
        lines.append({
            "id": len(lines) + 1, "compra_id": purchase_id, "producto_id": product_id,
            "cantidad": quantity, "precio_unitario": money(unit_price), "subtotal": money(subtotal),
            "pagado_con_puntos": redeemed,
        })
    balances[customer_id] += points_earned
    purchases.append({
        "id": purchase_id, "cliente_id": customer_id, "restaurante_id": restaurant_id,
        "fecha_hora": timestamp.isoformat(sep=" "), "monto_total": money(purchase_total),
        "forma_pago": rng.choice(["efectivo", "tarjeta"]),
    })
clean_tables = {
    "clientes": clients, "productos": products, "restaurantes": restaurants,
    "compras": purchases, "lineas_compra": lines,
}
print("Base válida:", {name: len(rows) for name, rows in clean_tables.items()})
print("Líneas canjeadas:", sum(line["pagado_con_puntos"] for line in lines))


Base válida: {'clientes': 1000, 'productos': 28, 'restaurantes': 10, 'compras': 20000, 'lineas_compra': 49525}
Líneas canjeadas: 2495


## 4. Validación de la base válida y de los patrones

Se comprueban esquemas, claves únicas, relaciones, fechas, montos, granularidad,
redondeo y saldos cronológicos. También se revisan los patrones de la ventana de
análisis, incluida la existencia de cuatro rezagados con índice de ventas menor a 0.5.
Estas verificaciones validan el generador; no sustituyen la limpieza de Silver.


In [5]:
def validate_clean_tables(tables):
    for name, rows in tables.items():
        assert all(list(row) == SCHEMAS[name] for row in rows), name
        assert len({row["id"] for row in rows}) == len(rows), f"ID repetido: {name}"
    customers = {row["id"]: row for row in tables["clientes"]}
    catalog = {row["id"]: row for row in tables["productos"]}
    stores = {row["id"] for row in tables["restaurantes"]}
    headers = {row["id"]: row for row in tables["compras"]}
    assert len({row["correo"] for row in customers.values()}) == len(customers)
    for row in customers.values():
        assert row["nombre"] and row["correo"]
        assert date.fromisoformat(row["fecha_registro"]) < PERIOD_START
    for row in catalog.values():
        assert row["categoria"] in CATEGORIES
        assert cents(row["precio_normal"]) > 0 and row["precio_en_puntos"] > 0
    grouped_lines = defaultdict(list)
    for line in tables["lineas_compra"]:
        assert line["compra_id"] in headers and line["producto_id"] in catalog
        assert isinstance(line["cantidad"], int) and line["cantidad"] >= 1
        assert isinstance(line["pagado_con_puntos"], bool)
        assert cents(line["subtotal"]) == line["cantidad"] * cents(line["precio_unitario"])
        assert cents(line["precio_unitario"]) >= 0
        if line["pagado_con_puntos"]:
            assert cents(line["precio_unitario"]) == 0
        else:
            assert cents(line["precio_unitario"]) == cents(catalog[line["producto_id"]]["precio_normal"])
        grouped_lines[line["compra_id"]].append(line)
    audited_balances = defaultdict(int)
    redeemed_lines = 0
    for purchase in sorted(headers.values(), key=lambda row: (row["fecha_hora"], row["id"])):
        customer_id = purchase["cliente_id"]
        assert customer_id in customers and purchase["restaurante_id"] in stores
        timestamp = datetime.fromisoformat(purchase["fecha_hora"])
        assert PERIOD_START <= timestamp.date() <= PERIOD_END
        assert time(6) <= timestamp.time() < time(23)
        assert date.fromisoformat(customers[customer_id]["fecha_registro"]) < timestamp.date()
        assert purchase["forma_pago"] in {"efectivo", "tarjeta"}
        details = grouped_lines[purchase["id"]]
        assert 1 <= len(details) <= 4
        assert len({line["producto_id"] for line in details}) == len(details)
        assert cents(purchase["monto_total"]) == sum(cents(line["subtotal"]) for line in details)
        assert cents(purchase["monto_total"]) >= 0
        earned = 0
        for line in details:
            if line["pagado_con_puntos"]:
                cost = catalog[line["producto_id"]]["precio_en_puntos"] * line["cantidad"]
                assert audited_balances[customer_id] >= cost, "Canje sin saldo previo"
                audited_balances[customer_id] -= cost
                redeemed_lines += 1
            else:
                earned += base_points(cents(line["subtotal"]))
        audited_balances[customer_id] += earned
    assert redeemed_lines > 0
    assert len(audited_balances) == 1000 and min(audited_balances.values()) >= 0
    return dict(audited_balances)


assert validate_clean_tables(clean_tables) == dict(balances)
# Los diagnósticos usan exclusivamente enero–febrero, como las futuras tablas Gold.
analysis_purchases = {row["id"]: row for row in purchases if date.fromisoformat(row["fecha_hora"][:10]) <= ANALYSIS_END}
analysis_units = Counter()
profile_category_visits = defaultdict(set)
for line in lines:
    purchase = analysis_purchases.get(line["compra_id"])
    if purchase is None:
        continue
    if not line["pagado_con_puntos"]:
        analysis_units[line["producto_id"]] += line["cantidad"]
    profile = hidden_profiles[purchase["cliente_id"]]
    profile_category_visits[(profile, product_by_id[line["producto_id"]]["categoria"])].add(purchase["id"])
laggard_indices = {}
for category, ids in products_by_category.items():
    category_average = sum(analysis_units[pid] for pid in ids) / len(ids)
    assert category_average > 0
    for product_id in ids:
        index = analysis_units[product_id] / category_average
        if index < 0.5:
            laggard_indices[product_id] = index
assert set(laggard_indices) == laggard_ids, laggard_indices
print("Rezagados en meses 1–2:")
for product_id, index in sorted(laggard_indices.items()):
    print(f"  {product_by_id[product_id]['nombre']}: índice {index:.3f}")

profile_stats = {}
for profile in PROFILES:
    rows = [row for row in purchases if hidden_profiles[row["cliente_id"]] == profile]
    monthly = [sum(row["fecha_hora"].startswith(f"{YEAR}-{month:02d}") for row in rows) for month in (1, 2, 3)]
    morning_share = sum(6 <= datetime.fromisoformat(row["fecha_hora"]).hour < 11 for row in rows) / len(rows)
    ticket = sum(cents(row["monto_total"]) for row in rows) / len(rows) / 100
    profile_stats[profile] = (monthly, morning_share, ticket)
    assert monthly == monthly_targets[profile]
    print(f"{profile:14s} | compras por mes {monthly} | mañana {morning_share:.1%} | ticket Q{ticket:.2f}")
assert profile_stats["Madrugadores"][1] > 0.90
assert profile_stats["En riesgo"][0][1] < profile_stats["En riesgo"][0][0] * 0.15
assert profile_stats["Leales"][2] > profile_stats["Ocasionales"][2]
occasional_counts = Counter((row["cliente_id"], row["fecha_hora"][:7]) for row in purchases if hidden_profiles[row["cliente_id"]] == "Ocasionales")
assert len(occasional_counts) == 750 and set(occasional_counts.values()) <= {1, 2}
# Afinidad observable en datos, sin exportar etiquetas de generación.
profile_analysis_counts = Counter(hidden_profiles[row["cliente_id"]] for row in analysis_purchases.values())
for category in CATEGORIES:
    all_visits = sum(len(profile_category_visits[(profile, category)]) for profile in PROFILES)
    overall_share = all_visits / len(analysis_purchases)
    assert overall_share > 0
    affinity = {
        profile: (len(profile_category_visits[(profile, category)]) / profile_analysis_counts[profile]) / overall_share
        for profile in PROFILES
    }
    assert max(affinity.values()) > 1.2, (category, affinity)
print("Validación correcta: integridad, importes, canjes, saldos y patrones de comportamiento.")


Rezagados en meses 1–2:
  Hamburguesa especial: índice 0.064
  Sándwich de pollo especial: índice 0.052
  Pie de manzana: índice 0.034
  Ensalada especial: índice 0.046
Leales         | compras por mes [3000, 3000, 3000] | mañana 24.4% | ticket Q102.07
Madrugadores   | compras por mes [2166, 2167, 2167] | mañana 97.3% | ticket Q38.67
En riesgo      | compras por mes [3000, 250, 250] | mañana 10.8% | ticket Q59.09
Ocasionales    | compras por mes [333, 333, 334] | mañana 9.1% | ticket Q27.11
Validación correcta: integridad, importes, canjes, saldos y patrones de comportamiento.


## 5. Errores controlados y exportación Bronze

Los originales válidos permanecen intactos. Se agregan 16 duplicados exactos y dos
compras adicionales: una con importe negativo y otra con una referencia a un producto
inexistente. Ambas tienen una línea asociada con identificador nuevo.

La compra que contiene el producto inexistente deberá revisarse junto con su línea en
Silver, para evitar un encabezado sin detalle válido. Esta fase no ejecuta esa limpieza.
Los CSV usan UTF-8, encabezados del diccionario de datos, decimales con punto y
booleanos `true`/`false`. Reejecutar sobrescribe estos cinco archivos.


In [6]:
bronze_tables = {name: [dict(row) for row in rows] for name, rows in clean_tables.items()}
duplicate_counts = {"clientes": 2, "productos": 2, "restaurantes": 2, "compras": 5, "lineas_compra": 5}
for name, count in duplicate_counts.items():
    bronze_tables[name].extend(dict(row) for row in clean_tables[name][:count])

negative_purchase_id = len(purchases) + 1
missing_product_purchase_id = len(purchases) + 2
negative_line_id = len(lines) + 1
missing_product_line_id = len(lines) + 2
for purchase_id, line_id, amount, product_id in [
    (negative_purchase_id, negative_line_id, "-12.35", 13),
    (missing_product_purchase_id, missing_product_line_id, "15.00", 999999),
]:
    bronze_tables["compras"].append({
        "id": purchase_id, "cliente_id": 1, "restaurante_id": 1,
        "fecha_hora": f"{YEAR}-03-31 22:59:59", "monto_total": amount, "forma_pago": "efectivo",
    })
    bronze_tables["lineas_compra"].append({
        "id": line_id, "compra_id": purchase_id, "producto_id": product_id, "cantidad": 1,
        "precio_unitario": amount, "subtotal": amount, "pagado_con_puntos": False,
    })

BRONZE.mkdir(parents=True, exist_ok=True)
for name, rows in bronze_tables.items():
    destination = BRONZE / f"{name}.csv"
    with destination.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=SCHEMAS[name])
        writer.writeheader()
        for row in rows:
            writer.writerow({key: str(value).lower() if isinstance(value, bool) else value for key, value in row.items()})
    print(f"{destination.name:20s}: {len(rows):6,d} filas")


clientes.csv        :  1,002 filas
productos.csv       :     30 filas
restaurantes.csv    :     12 filas


compras.csv         : 20,007 filas


lineas_compra.csv   : 49,532 filas


## 6. Auditoría de los archivos escritos

Se releen los archivos exportados para comprobar tipos representados, conteos,
duplicados y los dos defectos intencionales. Se reconstruye la base conocida válida
**solo en memoria para auditar la exportación**, y se vuelve a revisar el saldo histórico.
Los CSV de Bronze conservan todos los errores; no se escriben tablas Silver ni Gold.


In [7]:
integer_columns = {
    "clientes": {"id"}, "productos": {"id", "precio_en_puntos"}, "restaurantes": {"id"},
    "compras": {"id", "cliente_id", "restaurante_id"},
    "lineas_compra": {"id", "compra_id", "producto_id", "cantidad"},
}
loaded_tables = {}
for name in SCHEMAS:
    path = BRONZE / f"{name}.csv"
    with path.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == SCHEMAS[name]
        rows = list(reader)
    for row in rows:
        assert all(value is not None and value != "" for value in row.values())
        for column in integer_columns[name]:
            row[column] = int(row[column])
        if name == "lineas_compra":
            assert row["pagado_con_puntos"] in {"true", "false"}
            row["pagado_con_puntos"] = row["pagado_con_puntos"] == "true"
    loaded_tables[name] = rows
    assert rows == bronze_tables[name], f"La exportación alteró {name}"
    duplicates = len(rows) - len({tuple(row.values()) for row in rows})
    assert duplicates == duplicate_counts[name], (name, duplicates)
    extra_invalid = 2 if name in {"compras", "lineas_compra"} else 0
    assert len(rows) == len(clean_tables[name]) + duplicate_counts[name] + extra_invalid

negative_headers = [row["id"] for row in loaded_tables["compras"] if cents(row["monto_total"]) < 0]
negative_details = [row["id"] for row in loaded_tables["lineas_compra"] if cents(row["subtotal"]) < 0 or cents(row["precio_unitario"]) < 0]
unknown_products = [row["id"] for row in loaded_tables["lineas_compra"] if row["producto_id"] not in product_by_id]
assert negative_headers == [negative_purchase_id]
assert negative_details == [negative_line_id]
assert unknown_products == [missing_product_line_id]

# Reconstrucción de los originales conocidos para auditar el ciclo de escritura/lectura.
round_trip_clean = {}
for name, rows in loaded_tables.items():
    original_ids = {row["id"] for row in clean_tables[name]}
    round_trip_clean[name] = list({row["id"]: row for row in rows if row["id"] in original_ids}.values())
assert validate_clean_tables(round_trip_clean) == dict(balances)
print("Auditoría Bronze correcta: 16 duplicados y 2 compras con sus líneas inválidas.")
print(f"Saldo mínimo final bajo regla fija: {min(balances.values())} puntos.")
print("Huellas SHA-256 para comprobar reproducibilidad:")
for name in SCHEMAS:
    path = BRONZE / f"{name}.csv"
    print(f"  {path.name}: {hashlib.sha256(path.read_bytes()).hexdigest()}")


Auditoría Bronze correcta: 16 duplicados y 2 compras con sus líneas inválidas.
Saldo mínimo final bajo regla fija: 150 puntos.
Huellas SHA-256 para comprobar reproducibilidad:
  clientes.csv: 9db2f7cfb24bc221dfe8a30ff3bb9f949ba2960c48bb8099c6ce40a807f95841
  productos.csv: f1c4a4d9280f8c63c2dc0e9cf40974f885af1e5dc3e224e3a914b387de496acd
  restaurantes.csv: a976a4e05573cc3673268526e1f4396c4841ac1a363d728f3054330716008bab
  compras.csv: d2701a9b6f76ab974d90d526ab3700b5ecd296c94eaa9f0efb93b797e082b1e7
  lineas_compra.csv: dd4fb50b30a03d674b893b9adc3563b46f01dd47275f47e702cadc9eab00653e


## Resultado y siguiente fase

Bronze contiene las cinco fuentes sintéticas del documento. Las validaciones prueban
que el generador produce transacciones coherentes, canjes financiados por puntos
anteriores y patrones útiles para la segmentación posterior.

El notebook **02_bronze_a_silver** deberá limpiar con DuckDB, conservar los rechazados
en cuarentena y guardar las tablas válidas en Parquet. Los perfiles, modelos, reglas
dinámicas y movimientos finales quedan para los notebooks 03–05.
Los datos sintéticos no demuestran un aumento real de ventas.
